<a href="https://colab.research.google.com/github/ncsu-geoforall-lab/GIS582-assignments/blob/main/7AB%20-%20Flow%20Modeling/7A_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 7A: Flow Routing and Watershed Analysis

**Course:** [GIS 582 - Geospatial Modeling and Analysis](https://ncsu-geoforall-lab.github.io/geospatial-modeling-course/grass/hydrology.html)  
**Institution:** [NC State University, Center for Geospatial Analytics](https://cnr.ncsu.edu/geospatial/)
**Instructors:** Helena Mitasova, Corey White, and team

## Resources

- [GRASS overview and manual](https://grass.osgeo.org/grass84/manuals/index.html)

## Data

- Point coordinates: [outlet_point.txt](data/outlet_point.txt)

## Learning Objectives

In this tutorial, you will learn how to:
- Compute flow direction, flow accumulation and subwatersheds
- Create map of DEM depressions
- Derive contributing area for a given outlet
- Compute refined flow pattern using D-inf
- Compare multiple flow direction (MFD) with single flow direction (SFD) methods
- Compute wetness index
- Create a map of flooded area

## Tutorial Outline

* Part 1: Environment Setup
* Part 2: Compute flow direction, flow accumulation and subwatersheds
* Part 3: Create map of DEM depressions
* Part 4: Derive contributing area for a given outlet
* Part 5: Compute refined flow pattern using D-inf
* Part 6: Compare MFD and SFD flow accumulation
* Part 7: Compute wetness index
* Part 8: Create a map of flooded area
* Part 9: Optional - Assess and mitigate impact of the road on flow routing

---
## Part 1: Environment Setup

### Install GRASS

**Important:** This setup takes 3-5 minutes. You'll need to run it each time you start a new Colab session.

In [ ]:
!add-apt-repository -y ppa:ubuntugis/ubuntugis-unstable
!apt update
!apt-get install -y grass-core grass-dev

Check that GRASS is installed and available:

In [ ]:
!grass --version

Check which Python version is running:

In [ ]:
import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import GRASS Python packages

We need to locate GRASS Python packages using `grass --config python_path` and add them to `sys.path`.

In [ ]:
import subprocess
import os
from pathlib import Path

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj

### Download North Carolina Sample Dataset

In [ ]:
!grass --tmp-project XY --exec g.download.project url=https://grass.osgeo.org/sampledata/north_carolina/nc_spm_08_grass7.tar.gz path=/content

Download the outlet point coordinates file:

In [ ]:
!curl -L -o outlet_point.txt http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/outlet_point.txt

### Initialize GRASS Session

In [ ]:
# Start GRASS session
grassdata = "/content"
project = "nc_spm_08_grass7"
mapset = "user1"  # Create a new mapset for our work

# Start GRASS Session
session = gj.init(Path(project, mapset))
print("GRASS session started.")

---
## Part 2: Compute flow direction, flow accumulation and subwatersheds

Compute flow direction, flow accumulation and subwatersheds with approx. size of 10000 cells from 30m NED using least cost path method with single flow direction (SFD).

In [ ]:
gs.run_command("g.region", raster="elev_ned_30m", flags="p")
gs.run_command(
    "r.watershed",
    elevation="elev_ned_30m",
    threshold=10000,
    accumulation="accum_10K",
    drainage="draindir_10K",
    basin="basin_10K",
    flags="s",
)

Extract detailed streams from flow accumulation raster and convert basins and streams to vector format:

In [ ]:
gs.run_command("r.mapcalc", expression="streams_der_30m = if(abs(accum_10K) > 100, 1, null())")
gs.run_command("r.to.vect", input="basin_10K", output="basin_10K", type="area", flags="s")
gs.run_command("r.thin", input="streams_der_30m", output="streams_der_30m_t")
gs.run_command("r.to.vect", input="streams_der_30m_t", output="streams_der_30m", type="line", flags="s")

Generate relief shaded map of basins and display extracted streams along with the official Wake county streams (red).

In [ ]:
streams_map = gj.Map(filename="mystreams.png")
streams_map.d_his(hue="basin_10K", intensity="elevation_shade", brighten=40)
streams_map.d_vect(map="basin_10K", type="boundary")
streams_map.d_rast(map="lakes")
streams_map.d_vect(map="streams_der_30m", color="blue", width=2)
streams_map.show()

Compare the derived streams with the official stream map:

In [ ]:
streams_compare_map = gj.Map(filename="streams_compare.png")
streams_compare_map.d_his(hue="basin_10K", intensity="elevation_shade", brighten=40)
streams_compare_map.d_vect(map="basin_10K", type="boundary")
streams_compare_map.d_rast(map="lakes")
streams_compare_map.d_vect(map="streams_der_30m", color="blue", width=2)
streams_compare_map.d_vect(map="streams", color="red")
streams_compare_map.show()

Note that you can also use [r.stream.extract](https://grass.osgeo.org/grass-stable/manuals/r.stream.extract.html) for raster and vectorized streams with topology of stream network. Read the manual pages to learn more.

#### Question 2.1

How do the derived streams compare with the official stream map?

`Add text here with your answer to question 2.1`

#### Task 2.1

Modify the mapcalc expression to make stream origins fit better with the official stream map and include the modified expression and map in your report.

In [ ]:
# Add code here for task 2.1

#### Question 2.2

What do the negative values in the flow direction and flow accumulation maps mean? (hint: read the [r.watershed](https://grass.osgeo.org/grass-stable/manuals/r.watershed.html) manual page)

`Add text here with your answer to question 2.2`

---
## Part 3: Create map of DEM depressions

Depression filling is sometimes necessary for certain flow routing algorithms but it can significantly alter the elevation data leading to artificial straight channels. Find out how extensive the depressions are in our DEM.

Note that `r.watershed` doesn't need any depression filling thanks to its underlying algorithm which uses least cost path to get over depressions.

In [ ]:
gs.run_command("g.region", raster="elevation", flags="p")
gs.run_command("r.fill.dir", input="elevation", output="elev_fill1", direction="dir1", areas="unres1")
gs.run_command("r.fill.dir", input="elev_fill1", output="elev_fill2", direction="dir2", areas="unres2")
gs.run_command("r.fill.dir", input="elev_fill2", output="elev_fill3", direction="dir3", areas="unres3")
gs.run_command("r.mapcalc", expression="depr_bin = if((elevation-elev_fill3) < 0., 1, null())")
gs.run_command("r.colors", map="depr_bin", color="blues")

Display the depressions and compare them with actual lakes:

In [ ]:
depr_map = gj.Map(filename="depressions.png")
depr_map.d_rast(map="elevation")
depr_map.d_vect(map="roadsmajor")
depr_map.d_rast(map="depr_bin")
depr_map.d_vect(map="lakes", type="area", fill_color="blue")
depr_map.show()

In [ ]:
gs.parse_command("r.report", map="depr_bin", unit="h,p")

#### Question 3.1

What is the area mapped as depression in [ha] and [%]? Are all depressions in DEM real or artificial?

Optional: Compute the total volume of mapped depressions.

`Add text here with your answer to question 3.1`

---
## Part 4: Derive contributing area for a given outlet

Set region to the high resolution study area and compute flow accumulation, flow direction and basins with 15000 cells threshold:

In [ ]:
gs.run_command("g.region", region="rural_1m", flags="p")
gs.run_command(
    "r.watershed",
    elevation="elev_lid792_1m",
    threshold=15000,
    accumulation="accum_15K",
    drainage="draindir_15K",
    basin="basin_15K",
    flags="as",
)

Display extracted streams over aerial image:

In [ ]:
accum_map = gj.Map(filename="accum_15K.png")
accum_map.d_rast(map="ortho_2001_t792_1m")
accum_map.d_rast(map="accum_15K", values="1000-1000000")
accum_map.show()

Delineate contributing area (watershed) associated with an outlet located on the extracted stream and convert it to vector format. Import coordinates of the outlet to include it in the map.

In [ ]:
gs.run_command(
    "r.water.outlet",
    input="draindir_15K",
    output="basin_A30",
    coordinates="638845.52,220085.26",
)
gs.run_command("r.to.vect", input="basin_A30", output="basin_A30", type="area", flags="s")
gs.run_command("v.in.ascii", input="outlet_point.txt", output="outletA30", separator="space")

Display watershed boundary along with contours:

In [ ]:
gs.run_command("r.contour", input="elev_lid792_1m", output="elev_lid792_cont_1m", step=1, minlevel=104)

watershed_map = gj.Map(filename="watershedA30.png")
watershed_map.d_rast(map="ortho_2001_t792_1m")
watershed_map.d_vect(map="basin_A30", type="boundary", color="green", width=2)
watershed_map.d_vect(map="elev_lid792_cont_1m", color="white")
watershed_map.d_vect(map="outletA30", color="yellow", size=15, icon="basic/circle")
watershed_map.show()

Compute the watershed area:

In [ ]:
gs.parse_command("r.report", map="basin_A30", unit="h,a")

#### Question 4.1

What is the delineated watershed area [ha]? Compute average slope of this watershed: include the workflow and the resulting average (hint: check the previous assignments for workflow).

`Add text here with your answer to question 4.1`

In [ ]:
# Add code here for question 4.1

---
## Part 5: Compute refined flow pattern using D-inf

Compare upslope and downslope flow lines.

In [ ]:
gs.run_command("g.region", raster="elev_lid792_1m", flags="p")
gs.run_command(
    "r.flow",
    elevation="elev_lid792_1m",
    flowline="flowlines",
    flowlength="flowlg_1m",
    flowaccumulation="flowacc_1m",
)
gs.run_command(
    "r.flow",
    elevation="elev_lid792_1m",
    flowlength="flowlgup_1m",
    flowaccumulation="flowaccup_1m",
    flags="u",
)

Display maps along with contours to see the relation of flow pattern to terrain:

In [ ]:
flowdown_map = gj.Map(filename="flowdown.png")
flowdown_map.d_rast(map="flowacc_1m")
flowdown_map.d_vect(map="elev_lid792_cont_1m", color="red")
flowdown_map.d_vect(map="flowlines")
flowdown_map.show()

In [ ]:
flowup_map = gj.Map(filename="flowup.png")
flowup_map.d_rast(map="flowaccup_1m")
flowup_map.d_vect(map="elev_lid792_cont_1m", color="red")
flowup_map.show()

In [ ]:
flowlg_map = gj.Map(filename="flowlength.png")
flowlg_map.d_rast(map="flowlg_1m")
flowlg_map.d_vect(map="elev_lid792_cont_1m", color="red")
flowlg_map.d_legend(raster="flowlg_1m", at=(2, 40, 2, 6))
flowlg_map.show()

#### Question 5.1

On what type of landform (ridge, valley) do the upslope and downslope flowlines converge?

`Add text here with your answer to question 5.1`

---
## Part 6: Compare MFD and SFD flow accumulation

Compute flow accumulation using the default MFD method and display the flow accumulation maps from MFD and SFD (the Dinf map `flowacc_1m` was displayed above):

In [ ]:
gs.run_command("g.region", raster="elev_lid792_1m", flags="p")
gs.run_command(
    "r.watershed",
    elevation="elev_lid792_1m",
    threshold=15000,
    accumulation="accum_mfd_15K",
    flags="a",
)

In [ ]:
mfd_map = gj.Map(filename="stream_mfd.png")
mfd_map.d_rast(map="accum_mfd_15K")
mfd_map.d_vect(map="streams", color="red")
mfd_map.show()

In [ ]:
sfd_map = gj.Map(filename="stream_sfd.png")
sfd_map.d_rast(map="accum_15K")
sfd_map.d_vect(map="streams", color="red")
sfd_map.show()

#### Question 6.1

Compare the MFD and SFD results of r.watershed. Why are they different, which is more realistic and why?

`Add text here with your answer to question 6.1`

---
## Part 7: Compute wetness index

In [ ]:
gs.run_command("g.region", region="rural_1m", flags="p")
gs.run_command("r.topidx", input="elev_lid792_1m", output="wetness_1m")
gs.run_command("r.colors", map="wetness_1m", color="water")

In [ ]:
wetness_map = gj.Map(filename="wetness.png")
wetness_map.d_rast(map="wetness_1m")
wetness_map.d_legend(raster="wetness_1m", at=(2, 40, 2, 6))
wetness_map.show()

Optional: Compute the wetness index using the formula in the lecture (hint: you will need to compute slope using `elev_lid792_1m` and use `accum_mfd_15K` as contributing area). Check the values by comparing your output with `wetness_1m` computed with `r.topidx`.

Display `elev_lid792_1m` in 3D and drape over `wetness_1m` as color.

---
## Part 8: Create a map of flooded area

Create a map of flooded area for a given water level and seed point:

In [ ]:
gs.run_command("g.region", raster="elev_lid792_1m", flags="p")
gs.run_command(
    "r.lake",
    elevation="elev_lid792_1m",
    water_level=113.5,
    lake="flood1",
    coordinates="638728,220278",
)

In [ ]:
flood_map = gj.Map(filename="floodedarea.png")
flood_map.d_rast(map="elev_lid792_1m")
flood_map.d_rast(map="flood1")
flood_map.show()

#### Question 8.1

Increase water level to 113.7m and 115.0m and create flooded area maps at these two levels. Include the maps of flooded areas at these water levels in your report. Why is the flooded area much larger than for 113.5m?

`Add text here with your answer to question 8.1`

In [ ]:
# Add code here for question 8.1

---
## Part 9: Optional - Assess and mitigate impact of the road on flow routing

Compare the extracted streams (accum > 1500) with official stream data:

In [ ]:
gs.run_command("g.region", raster="elev_lid792_1m", flags="p")

stream_compare_map = gj.Map(filename="streamcompare.png")
stream_compare_map.d_rast(map="ortho_2001_t792_1m")
stream_compare_map.d_rast(map="accum_15K", values="1500-1000000")
stream_compare_map.d_vect(map="streets_wake", color="red")
stream_compare_map.d_vect(map="streams", color="green")
stream_compare_map.show()

Carve a pre-defined channel given by the stream data into DEM:

In [ ]:
gs.run_command(
    "r.carve",
    raster="elev_lid792_1m",
    vector="streams",
    width=2,
    depth=0.8,
    output="elev_lidcarved_1m",
)
gs.run_command("r.colors", map="elev_lidcarved_1m", raster="elev_lid792_1m")

In [ ]:
carved_map = gj.Map(filename="elev_lidcarved.png")
carved_map.d_rast(map="elev_lidcarved_1m")
carved_map.show()

Extract streams from the carved DEM and compare with the official streams map:

In [ ]:
gs.run_command(
    "r.watershed",
    elevation="elev_lidcarved_1m",
    accumulation="accumc_5K1m",
    flags="as",
)

In [ ]:
streamcarved_map = gj.Map(filename="streamcarved.png")
streamcarved_map.d_rast(map="accumc_5K1m", values="1500-10000000")
streamcarved_map.d_vect(map="streams")
streamcarved_map.show()

#### Question 9.1

What is the difference between `accum_5K`, `accumc_5K1m` and `streams`? Explain the advantage and disadvantage of carving.

`Add text here with your answer to question 9.1`